# 🏭 AI-Enabled Assets Performance & Predictive Maintenance Platform

This notebook allows you to run the full **Industrial Risk AI** platform directly in Google Colab.


### Step 1: 🚀 Setup Environment
Run this cell to install the platform and dependencies.

In [ ]:
import os
import subprocess
import time
import sys

# 1. Clone Repository
REPO_URL = "https://github.com/lmudu2/industrial-risk-ai.git"
REPO_DIR = "industrial-risk-ai"

if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

print(f"Cloning {REPO_URL}...")
!git clone {REPO_URL}
os.chdir(f"/content/{REPO_DIR}")

# 2. Install Dependencies
print("Installing dependencies (this may take 2 minutes)...")
!pip install -v -r requirements.txt

import streamlit
print(f"\n✅ Installed Streamlit version: {streamlit.__version__}")
if streamlit.__version__ < "1.34.0":
    print("❌ ERROR: Streamlit version is too low for this project.")
    print("👉 GO TO MENU: 'Runtime' -> 'Restart Session' and run this cell again!")
else:
    print("✅ Version check passed!")

### Step 2: 📊 Generate Industrial Database
Run this cell to generate the synthesized SQLite database.

In [ ]:
if not os.path.exists("backend/eam_database.db"):
    print("Generating real-world industrial data (3-5 minutes)...")
    !python data/generate_data.py
else:
    print("✅ Database already exists.")

### Step 3: ⚡ Start & Monitor Logs
This cell starts the platform and displays **Live Error Logs** below.

In [ ]:
# 2. Start Cloudflare Tunnel
import subprocess, time, threading, re, os

print("Opening Secure Cloudflare Tunnel...")
if not os.path.exists("cloudflared"):
    print("Downloading cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# Kill any old tunnels
!pkill cloudflared

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"], 
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

print("Waiting for Cloudflare URL (this can take 30s)...\n")
url = "NOT_FOUND"
# Wait up to 60 seconds
for _ in range(60):
    line = tunnel.stdout.readline()
    if not line: break
    if ".trycloudflare.com" in line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            print("\n" + "*"*60)
            print(f"🚀 SECURE DASHBOARD LINK: {url}")
            print("*"*60 + "\n")
            break
    time.sleep(0.1)

if url == "NOT_FOUND":
    print("❌ ERROR: Cloudflare Tunnel failed to provide a URL.")
    print("Try running the cell again.")

print("--- LIVE STREAMLIT LOGS ---")
def tail_logs():
    if not os.path.exists("streamlit.log"): return
    with open("streamlit.log", "r") as f:
        while True:
            line = f.readline()
            if line:
                # Strict noise filter
                junk = ["Local URL:", "Network URL:", "External URL:", "  http://", "view your Streamlit", "statistics"]
                if not any(x in line for x in junk):
                    print(line, end="")
            else:
                time.sleep(1)

threading.Thread(target=tail_logs, daemon=True).start()

try:
    while True: time.sleep(60)
except:
    print("Stopping...")
